In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

GRID_CSV_PATH = Path("1km_grid.csv")


def load_grid_table(csv_path: str | Path = GRID_CSV_PATH) -> pd.DataFrame:
    grid_df = pd.read_csv(csv_path)
    required_cols = {
        "grid_id", "x_min", "x_max", "y_min", "y_max", "x_center", "y_center"
    }
    missing_cols = required_cols - set(grid_df.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {sorted(missing_cols)}")
    return grid_df


def get_grid_coordinates(grid_df: pd.DataFrame, grid_id: int) -> dict:
    row = grid_df.loc[grid_df["grid_id"] == grid_id]
    if row.empty:
        raise ValueError(f"grid_id={grid_id} was not found")

    row = row.iloc[0]
    return {
        "grid_id": int(row["grid_id"]),
        "west": float(row["x_min"]),
        "east": float(row["x_max"]),
        "south": float(row["y_min"]),
        "north": float(row["y_max"]),
        "center_lng": float(row["x_center"]),
        "center_lat": float(row["y_center"]),
    }


grid_df = load_grid_table()
display(grid_df.head())

# Change this to the grid you want to inspect.
TARGET_GRID_ID = 0
grid_coords = get_grid_coordinates(grid_df, TARGET_GRID_ID)
grid_coords


,grid_id,meo_id,x_min,x_max,y_min,y_max,x_center,y_center,B,T,L,R,RB,RT,LB,LT
0,0,1,120.026858,120.036681,23.066442,23.075531,120.031770,23.070986,-1,1,-1,2,-1,3,-1,-1
1,1,1,120.026793,120.036617,23.075471,23.084560,120.031705,23.080016,0,-1,-1,3,2,4,-1,-1
2,2,1,120.036617,120.046440,23.066501,23.075590,120.041528,23.071046,-1,3,0,9,8,10,-1,1
3,3,1,120.036553,120.046376,23.075531,23.084619,120.041464,23.080075,2,4,1,10,9,11,0,-1
4,4,1,120.036488,120.046312,23.084560,23.093649,120.041400,23.089105,3,5,-1,11,10,12,1,-1


{'grid_id': 0,
 'west': 120.02685813718512,
 'east': 120.03668140607375,
 'south': 23.06644171298762,
 'north': 23.075530872833003,
 'center_lng': 120.03176977162944,
 'center_lat': 23.070986292910312}

In [8]:
import json
import random
from pathlib import Path

import pandas as pd
import requests

try:
    import mapillary.interface as mly
except ImportError as exc:
    raise ImportError(
        "Please install dependencies first: pip install mapillary requests pandas"
    ) from exc


MAPILLARY_ACCESS_TOKEN = "MLY|26199948926374186|6135d688b588deeabb91aedf98e975bc"
OUTPUT_DIR = Path("mapillary_downloads")


def _to_dict(payload):
    if isinstance(payload, str):
        return json.loads(payload)
    return payload


def _image_properties(image_meta: dict) -> dict:
    if isinstance(image_meta, dict) and isinstance(image_meta.get("properties"), dict):
        return image_meta["properties"]
    return image_meta if isinstance(image_meta, dict) else {}


def _image_url_from_metadata(image_meta: dict) -> str | None:
    props = _image_properties(image_meta)
    for key in ["thumb_original_url", "thumb_2048_url", "thumb_1024_url", "thumb_256_url"]:
        if props.get(key):
            return props[key]
    return None


def search_mapillary_images_in_grid(
    grid_df: pd.DataFrame,
    grid_id: int,
    sample_size: int,
    access_token: str,
    image_type: str = "flat",
    organization_id: str | None = None,
    min_captured_at: str | None = None,
    max_captured_at: str | None = None,
    random_seed: int | None = 42,
):
    if not access_token or access_token == "REPLACE_WITH_YOUR_MAPILLARY_ACCESS_TOKEN":
        raise ValueError("Please replace MAPILLARY_ACCESS_TOKEN with a valid token")

    bbox = get_grid_coordinates(grid_df, grid_id)
    bbox_query = {
        "west": bbox["west"],
        "south": bbox["south"],
        "east": bbox["east"],
        "north": bbox["north"],
    }

    mly.set_access_token(access_token)

    filters = {}
    if image_type:
        filters["image_type"] = image_type
    if organization_id:
        filters["organization_id"] = organization_id
    if min_captured_at:
        filters["min_captured_at"] = min_captured_at
    if max_captured_at:
        filters["max_captured_at"] = max_captured_at

    raw_result = mly.images_in_bbox(bbox=bbox_query, **filters)
    result = _to_dict(raw_result)
    features = result.get("features", [])

    if not features:
        print(f"No Mapillary images matched grid_id={grid_id}")
        return []

    rng = random.Random(random_seed)
    selected_features = rng.sample(features, k=min(sample_size, len(features)))
    print(f"Found {len(features)} candidates and randomly selected {len(selected_features)} images")
    return selected_features


def download_mapillary_images(
    selected_features: list,
    access_token: str,
    output_dir: str | Path,
    grid_id: int,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    mly.set_access_token(access_token)

    records = []
    for idx, feature in enumerate(selected_features, start=1):
        props = feature.get("properties", {})
        image_id = str(props.get("id") or feature.get("id"))

        image_meta = _to_dict(
            mly.image_from_key(
                key=image_id,
                fields=[
                    "captured_at",
                    "thumb_original_url",
                    "thumb_2048_url",
                    "thumb_1024_url",
                    "thumb_256_url",
                    "computed_geometry",
                    "geometry",
                ],
            )
        )

        image_url = _image_url_from_metadata(image_meta)
        if not image_url:
            try:
                image_url = mly.image_thumbnail(image_id=image_id, resolution=2048)
            except Exception:
                image_url = None

        if not image_url:
            print(f"Skipped image_id={image_id} because no thumbnail URL was returned")
            continue

        response = requests.get(image_url, timeout=60)
        response.raise_for_status()

        save_path = output_dir / f"grid_{grid_id}_{idx:03d}_{image_id}.jpg"
        save_path.write_bytes(response.content)

        props = _image_properties(image_meta)
        geometry = props.get("computed_geometry") or image_meta.get("geometry") or props.get("geometry") or {}
        coordinates = geometry.get("coordinates", [None, None])

        records.append(
            {
                "grid_id": grid_id,
                "image_id": image_id,
                "captured_at": props.get("captured_at") or image_meta.get("captured_at"),
                "longitude": coordinates[0],
                "latitude": coordinates[1],
                "download_url": image_url,
                "save_path": str(save_path),
            }
        )
        print(f"[{idx}/{len(selected_features)}] Downloaded {save_path.name}")

    return pd.DataFrame(records)


# Change these two values before running.
TARGET_GRID_ID = 0
TARGET_IMAGE_COUNT = 5

selected_features = search_mapillary_images_in_grid(
    grid_df=grid_df,
    grid_id=TARGET_GRID_ID,
    sample_size=TARGET_IMAGE_COUNT,
    access_token=MAPILLARY_ACCESS_TOKEN,
    image_type="flat",  # You can change this to "pano" or "all".
    random_seed=42,
)

download_df = download_mapillary_images(
    selected_features=selected_features,
    access_token=MAPILLARY_ACCESS_TOKEN,
    output_dir=OUTPUT_DIR / f"grid_{TARGET_GRID_ID}",
    grid_id=TARGET_GRID_ID,
)

download_df


Requesting GET to https://tiles.mapillary.com/maps/vtp/mly1_public/2/14/13654/7112/?access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 357ms
Requesting GET to https://tiles.mapillary.com/maps/vtp/mly1_public/2/14/13655/7112/?access_token=MLY%7C26199948926374186%7C6135d688b588deeabb91aedf98e975bc
Response 200 OK received in 266ms
No Mapillary images matched grid_id=0


""
